# Análise de Vendas — Online Retail

## Objetivo do Projeto

Analisar os dados de vendas do *Online Retail* com Pandas para consolidar os pedidos, calcular o ticket médio e identificar os principais países, produtos e clientes em faturamento e volume de vendas.


## Fonte dos Dados

- **Dataset:** Online Retail
- **Fonte:** UCI Machine Learning Repository
- **Período dos dados:** 01/12/2010 a 09/12/2011
- **Descrição:** dados de transações de uma empresa de varejo online sediada no Reino Unido
- **Moeda:** libra esterlina (£)
- **Licença:** CC BY 4.0

[https://archive.ics.uci.edu/dataset/352/online+retail](https://archive.ics.uci.edu/dataset/352/online+retail)


## Carregamento dos Dados

O arquivo Excel é carregado por um caminho relativo, mantendo o notebook portátil para GitHub.


In [9]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")

display(df.head())


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Limpeza e Preparação

São mantidas apenas vendas com quantidade e preço positivos. Registros duplicados são removidos antes do cálculo do valor total de cada item.


In [10]:
# Mantém somente vendas válidas e remove duplicatas exatas.
df_projeto = (
    df.copy()
    .query("Quantity > 0 and UnitPrice > 0")
    .drop_duplicates()
)

df_projeto["Total_venda"] = df_projeto["UnitPrice"] * df_projeto["Quantity"]

# Cria colunas de apoio a partir da data do pedido.
df_projeto["Ano"] = df_projeto["InvoiceDate"].dt.year
df_projeto["Mes"] = df_projeto["InvoiceDate"].dt.month
df_projeto["Dia"] = df_projeto["InvoiceDate"].dt.day
df_projeto["Hora"] = df_projeto["InvoiceDate"].dt.hour

display(df_projeto.head())


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total_venda,Ano,Mes,Dia,Hora
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8


## Construção da Base por Pedido

As linhas de produtos são agrupadas por pedido para reunir faturamento, variedade de produtos, quantidade de itens, cliente, país e data.


In [11]:
base_pedidos = (
    df_projeto.copy()
    .groupby("InvoiceNo")
    .agg(
        faturamento_pedido=("Total_venda", "sum"),
        produtos_distintos=("StockCode", "nunique"),
        quantidade_itens=("Quantity", "sum"),
        cliente=("CustomerID", "first"),
        pais=("Country", "first"),
        data_pedido=("InvoiceDate", "first"),
    )
    .reset_index()
)

base_pedidos.index += 1

print(f"Dimensões da base por pedido: {base_pedidos.shape}")

base_pedidos_exibicao = base_pedidos.head(10).copy()
base_pedidos_exibicao["faturamento_pedido"] = base_pedidos_exibicao["faturamento_pedido"].map(
    lambda valor: f"£ {valor:.2f}"
)
base_pedidos_exibicao["cliente"] = base_pedidos_exibicao["cliente"].astype("Int64")

display(base_pedidos_exibicao)


Dimensões da base por pedido: (19960, 7)


,InvoiceNo,faturamento_pedido,produtos_distintos,quantidade_itens,cliente,pais,data_pedido
1,536365,£ 139.12,7,40,17850,United Kingdom,2010-12-01 08:26:00
2,536366,£ 22.20,2,12,17850,United Kingdom,2010-12-01 08:28:00
3,536367,£ 278.73,12,83,13047,United Kingdom,2010-12-01 08:34:00
4,536368,£ 70.05,4,15,13047,United Kingdom,2010-12-01 08:34:00
5,536369,£ 17.85,1,3,13047,United Kingdom,2010-12-01 08:35:00
6,536370,£ 855.86,20,449,12583,France,2010-12-01 08:45:00
7,536371,£ 204.00,1,80,13748,United Kingdom,2010-12-01 09:00:00
8,536372,£ 22.20,2,12,17850,United Kingdom,2010-12-01 09:01:00
9,536373,£ 259.86,16,88,17850,United Kingdom,2010-12-01 09:02:00
10,536374,£ 350.40,1,32,15100,United Kingdom,2010-12-01 09:09:00


## Ticket Médio

O ticket médio geral representa o faturamento médio por pedido.


In [12]:
ticket_medio_geral = base_pedidos["faturamento_pedido"].mean()

print(f"Ticket médio geral: £ {ticket_medio_geral:.2f}")


Ticket médio geral: £ 533.17


## Análise por País

O ranking abaixo apresenta os dez países com maior ticket médio por pedido.


In [13]:
top_10_paises = (
    base_pedidos.groupby("pais")
    .agg(ticket_medio=("faturamento_pedido", "mean"))
    .sort_values(by="ticket_medio", ascending=False)
    .reset_index()
    .head(10)
)

top_10_paises.index += 1
top_10_paises.index.name = "ranking"

top_10_paises_exibicao = top_10_paises.copy()
top_10_paises_exibicao["ticket_medio"] = top_10_paises_exibicao["ticket_medio"].map(
    lambda valor: f"£ {valor:.2f}"
)

display(top_10_paises_exibicao)


,pais,ticket_medio
ranking,,
1,Singapore,£ 3039.90
2,Netherlands,£ 3036.66
3,Australia,£ 2429.01
4,Japan,£ 1969.28
5,Lebanon,£ 1693.88
6,Hong Kong,£ 1407.55
7,Brazil,£ 1143.60
8,Sweden,£ 1065.77
9,Switzerland,£ 1056.81


## Produtos Mais Vendidos

Os produtos são ordenados pela quantidade total de unidades vendidas.


In [14]:
unidades_vendidas = (
    df_projeto.groupby(["StockCode", "Description"])
    .agg(
        unidades_vendidas=("Quantity", "sum"),
        faturamento=("Total_venda", "sum"),
        pedidos=("InvoiceNo", "nunique"),
    )
    .sort_values(by="unidades_vendidas", ascending=False)
    .reset_index()
)

unidades_vendidas.index += 1

unidades_vendidas_exibicao = unidades_vendidas.head(10).copy()
unidades_vendidas_exibicao["faturamento"] = unidades_vendidas_exibicao[
    "faturamento"
].map(lambda valor: f"£ {valor:.2f}")

display(unidades_vendidas_exibicao)


,StockCode,Description,unidades_vendidas,faturamento,pedidos
1,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,£ 168469.60,1
2,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,£ 81700.92,247
3,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54951,£ 13814.01,535
4,85099B,JUMBO BAG RED RETROSPOT,48371,£ 94159.81,2089
5,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,£ 104284.24,2189
6,22197,POPCORN HOLDER,36749,£ 34288.67,803
7,21212,PACK OF 72 RETROSPOT CAKE CASES,36396,£ 21246.45,1320
8,84879,ASSORTED COLOUR BIRD ORNAMENT,36362,£ 58927.62,1455
9,23084,RABBIT NIGHT LIGHT,30739,£ 66870.03,994
10,22492,MINI PAINT SET VINTAGE,26633,£ 16937.82,380


## Clientes com Maior Faturamento

Para esta análise, registros sem `CustomerID` são desconsiderados, pois não podem ser atribuídos a um cliente identificado.


In [15]:
resumo_receita = (
    df_projeto.dropna(subset=["CustomerID"])
    .groupby("CustomerID")
    .agg(
        pedidos_unicos=("InvoiceNo", "nunique"),
        unidades_compradas=("Quantity", "sum"),
        faturamento_total=("Total_venda", "sum"),
    )
    .reset_index()
    .sort_values(by="faturamento_total", ascending=False)
    .reset_index(drop=True)
)

resumo_receita.index += 1

resumo_receita_exibicao = resumo_receita.head(10).copy()
resumo_receita_exibicao["CustomerID"] = resumo_receita_exibicao[
    "CustomerID"
].astype("Int64")
resumo_receita_exibicao["faturamento_total"] = resumo_receita_exibicao[
    "faturamento_total"
].map(lambda valor: f"£ {valor:.2f}")

display(resumo_receita_exibicao)


,CustomerID,pedidos_unicos,unidades_compradas,faturamento_total
1,14646,73,196915,£ 280206.02
2,18102,60,64124,£ 259657.30
3,17450,46,69973,£ 194390.79
4,16446,2,80997,£ 168472.50
5,14911,201,80240,£ 143711.17
6,12415,21,77374,£ 124914.53
7,14156,55,57768,£ 117210.08
8,17511,31,64549,£ 91062.38
9,16029,63,40108,£ 80850.84
10,12346,1,74215,£ 77183.60


## Investigação de Outlier

O produto com 80.995 unidades vendidas aparece em um único pedido. O registro é mantido e apresentado apenas como um pedido atípico, sem atribuir uma causa que não possa ser comprovada pelos dados.


In [16]:
pedido_atipico = df_projeto.query("StockCode == 23843")

pedido_atipico_exibicao = pedido_atipico.copy()
pedido_atipico_exibicao["UnitPrice"] = pedido_atipico_exibicao["UnitPrice"].map(
    lambda valor: f"£ {valor:.2f}"
)
pedido_atipico_exibicao["CustomerID"] = pedido_atipico_exibicao["CustomerID"].astype("Int64")
pedido_atipico_exibicao["Total_venda"] = pedido_atipico_exibicao["Total_venda"].map(
    lambda valor: f"£ {valor:.2f}"
)

display(pedido_atipico_exibicao)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total_venda,Ano,Mes,Dia,Hora
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,£ 2.08,16446,United Kingdom,£ 168469.60,2011,12,9,9


## Resumo Executivo

Com base na análise dos dados de varejo online, foram identificados os seguintes pontos:

- **Ticket médio geral:** £ 533,17 por pedido.
- **Maior ticket médio por país:** Singapura — £ 3.039,90 por pedido.
- **Produto mais vendido:** PAPER CRAFT, LITTLE BIRDIE — 80.995 unidades.
- **Cliente com maior faturamento:** ID 14646 — £ 280.206,02.


## Considerações sobre a Qualidade dos Dados

Durante a preparação dos dados:

1. Os valores nulos em `CustomerID` foram desconsiderados somente na etapa de clientes, pois prejudicariam a **análise por cliente**.
2. As linhas duplicadas foram removidas para evitar que quantidades e valores fossem somados mais de uma vez.
3. Quantidades e preços não positivos foram filtrados, pois podem representar devoluções, cancelamentos ou registros não comerciais.
